In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import transformers

/opt/anaconda3/envs/mlops/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
resume_text = """
Education: Master of Computer Science, NYU
Experience: 2 years software engineer at XYZ Corp, backend development in Java and Python.
Skills: Java, Python, SQL, Docker, AWS
"""

In [ ]:
prompt = f"""
You are an experienced hiring manager.

Here is the candidate's resume:
{resume_text}

Instructions:
- ONLY analyze what is explicitly written in the resume.
- DO NOT invent or assume any experiences, skills, or details that are not present in the text.
- If information is missing, say "Not provided".
- Follow the structure below in your response.

Your review must include:

1. **Overall Evaluation**: A short summary of the strengths and weaknesses of the entire resume.  

2. **Section-by-Section Feedback**:  
   For each section of the resume (e.g., Education, Experience, Skills, Projects, etc.), do the following:  
   - Give a clear evaluation of that section.  
   - Provide specific improvement suggestions.  
   - If relevant, suggest how one entry could be rewritten for clarity or impact.  

Do not add any new sections that are not present in the resume.

"""

In [25]:
def ask_model(prompt, max_new_tokens=1000):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9
        )
    # 只取生成部分
    input_len = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return answer.strip()

In [26]:
print(ask_model(prompt))

---

**Summary:** The candidate has a solid background in computer science with experience in backend development using Java and Python. They have also acquired expertise in database management systems through their work at XYZ Corp. Their proficiency in cloud computing platforms like Docker and AWS further enhances their capabilities. 

**Evaluation:**
The candidate possesses strong foundational knowledge in computer science. With extensive experience in backend development, they possess both practical and theoretical understanding. Their experience working on projects that required efficient and scalable codebases showcases their ability to think logically and creatively while solving problems. 

However, there is room for improvement in the areas of project management and communication. As a software engineer, it would be beneficial if they were more organized in managing multiple tasks efficiently and communicated effectively during meetings. 

One entry could be rewritten for clar

In [ ]:
import onnxruntime as ort
from transformers import AutoTokenizer

# 1. 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

# 2. 构建 QNN Session
so = ort.SessionOptions()
providers = [("QNNExecutionProvider", {}), "CPUExecutionProvider"]

session = ort.InferenceSession("qwen25-onnx/model.onnx", sess_options=so, providers=providers)

# 3. 构建输入
prompt = """You are an AI resume reviewer. Please evaluate the following resume strictly 
based only on its content. Provide:
1. An overall assessment
2. Section-by-section feedback and improvement suggestions.

Resume:
---
Name: John Doe
Experience: Software Engineer at XYZ Corp (2020-2023)
Education: B.S. in Computer Science, ABC University
Skills: Python, Machine Learning, Cloud Computing
---
"""

inputs = tokenizer(prompt, return_tensors="np")

# 4. 推理 (decoder-only模型通常只要 input_ids)
onnx_inputs = {k: v for k, v in inputs.items() if k in [inp.name for inp in session.get_inputs()]}
outputs = session.run(None, onnx_inputs)

# 5. 解码
decoded = tokenizer.decode(outputs[0][0], skip_special_tokens=True)
print("=== Resume Feedback ===")
print(decoded)
